In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

# Obtenir la session active dans le Notebook
session = get_active_session()

# Configurer la base de données et le schéma (S'assurer qu'ils existent)
session.sql("CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.RAW_DATA").collect()
session.use_database("HOUSE_PRICE_DB")
session.use_schema("RAW_DATA")

print(f"Connecté à : {session.get_current_database()}.{session.get_current_schema()}")

# Phase 1 : Data Engineering et Ingestion
**Objectif :** Charger les données depuis S3 et préparer le dataset pour le modèle de ML.

### Identification des variables :
*   **Variable cible (y) :** `PRICE` (Prix de vente).
*   **Features (X) :** `AREA`, `BEDROOMS`, `BATHROOMS`, `STORIES`, `MAINROAD`, `GUESTROOM`, `BASEMENT`, `HOTWATERHEATING`, `AIRCONDITIONING`, `PARKING`, `PREFAREA`, `FURNISHINGSTATUS`.

In [ ]:
import snowflake.snowpark.functions as F

# 1. Créer le Stage (Configuration de la connexion S3)
session.sql("CREATE OR REPLACE STAGE house_price_stage URL = 's3://logbrain-datalake/datasets/house_price/'").collect()

# 2. Lire le fichier JSON brut
df_raw = session.read.json("@house_price_stage")

# 3. "APLATIR" (FLATTEN) la liste JSON pour que chaque maison soit une ligne
# C'est l'étape clé : nous convertissons le tableau [$1] en lignes individuelles
df_flattened = df_raw.flatten(F.col("$1"))

# 4. Extraire les champs depuis la colonne 'VALUE' (générée par le flatten)
# Important : Les clés dans votre fichier sont en MINUSCULES ("price", "area"...)
df_final = df_flattened.select(
    F.col("VALUE")["price"].cast("int").alias("PRICE"),
    F.col("VALUE")["area"].cast("int").alias("AREA"),
    F.col("VALUE")["bedrooms"].cast("int").alias("BEDROOMS"),
    F.col("VALUE")["bathrooms"].cast("int").alias("BATHROOMS"),
    F.col("VALUE")["stories"].cast("int").alias("STORIES"),
    F.col("VALUE")["mainroad"].cast("string").alias("MAINROAD"),
    F.col("VALUE")["guestroom"].cast("string").alias("GUESTROOM"),
    F.col("VALUE")["basement"].cast("string").alias("BASEMENT"),
    F.col("VALUE")["hotwaterheating"].cast("string").alias("HOTWATERHEATING"),
    F.col("VALUE")["airconditioning"].cast("string").alias("AIRCONDITIONING"),
    F.col("VALUE")["parking"].cast("int").alias("PARKING"),
    F.col("VALUE")["prefarea"].cast("string").alias("PREFAREA"),
    F.col("VALUE")["furnishingstatus"].cast("string").alias("FURNISHINGSTATUS")
)

# 5. Enregistrer dans la table finale (HOUSE_DATA)
df_final.write.mode("overwrite").save_as_table("HOUSE_DATA")

print("Succès absolu ! Les données ont été aplaties et chargées correctement.")

# 6. Vérification : Obtenir le nombre total de lignes et afficher les 50 premières
print(f"Nombre total de maisons chargées dans la table : {session.table('HOUSE_DATA').count()}")
session.table("HOUSE_DATA").limit(50).show()

In [ ]:
import snowflake.snowpark.functions as F

# Chargement de la table créée dans la cellule précédente
df = session.table("HOUSE_DATA")

print("--- 1. Statistiques descriptives (Moyennes, Min, Max) ---")
# Ceci affiche un tableau avec la moyenne, l'écart-type, etc.
df.describe().show()

print("\n--- 2. Vérification des valeurs nulles (Qualité des données) ---")
# Comptage du nombre de valeurs nulles dans chaque colonne de manière dynamique
null_summary = df.select([F.count(F.when(F.col(c).is_null(), c)).alias(c) for c in df.columns])
null_summary.show()

print(f"\n--- 3. Nombre total d'enregistrements analysés : {df.count()} ---")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Conversion des données de Snowflake vers Pandas pour la visualisation
# Remarque : Snowpark effectue le traitement lourd, Pandas reçoit uniquement les données pour l'affichage graphique
pdf = session.table("HOUSE_DATA").to_pandas()

# Graphique 1 : Distribution du Prix (Variable Cible)
plt.figure(figsize=(10, 5))
sns.histplot(pdf['PRICE'], kde=True, color='skyblue')
plt.title('Distribution des prix de vente')
plt.xlabel('Prix')
plt.ylabel('Fréquence')
plt.show()

# Graphique 2 : Matrice de Corrélation (Très important pour le professeur !)
# Cela permet d'identifier les variables (comme AREA ou BEDROOMS) qui influencent le plus le prix
plt.figure(figsize=(12, 8))
# Calcul de la corrélation uniquement pour les colonnes numériques
numeric_cols = pdf.select_dtypes(include=['number'])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matrice de Corrélation (Facteurs influençant le prix)')
plt.show()

# Graphique 3 : Relation Surface vs Prix
plt.figure(figsize=(10, 5))
sns.scatterplot(data=pdf, x='AREA', y='PRICE', hue='AIRCONDITIONING', alpha=0.6)
plt.title('Influence de la Surface (AREA) et de la Climatisation sur le Prix')
plt.show()